In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, AutoARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from src.util.data_path import fig_model_forecast, fig_model_error
(fig_model_forecast / "ARIMA").mkdir(parents=True, exist_ok=True)
(fig_model_error / "ARIMA").mkdir(parents=True, exist_ok=True)

In [ ]:
from util.config import CFG
from util.data_path import corn_long as data_file
from util.data_utils import load_and_prepare

train, future, series, scaler, long = load_and_prepare(data_file)
y_train = train["price"].asfreq("MS")
y_test = future["price"].asfreq("MS")

gregorian_2567 = CFG["train_cutoff_year"] + 1

# Setup sf object for EDA diagnostics (unfitted)
sf_data = (
    y_train.rename("y")
    .to_frame()
    .assign(ds=lambda df: df.index, unique_id=data_file.parent.name)
    .reset_index(drop=True)
)
sf = StatsForecast(
    models=[AutoARIMA(season_length=12, stepwise=True)],
    freq="MS",
)


In [ ]:
# Add diagnostics to check model parameters
try:
    model_info = sf.models[0].model_
    print("Selected ARIMA Model Information:")
    print(model_info)
except AttributeError:
    print("Model information not available in this version of statsforecast")

# Plot ACF and PACF to understand time series characteristics
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(y_train, lags=24, ax=ax1)
ax1.set_title("Autocorrelation Function (ACF)")
plot_pacf(y_train, lags=24, ax=ax2)
ax2.set_title("Partial Autocorrelation Function (PACF)")
plt.tight_layout()
plt.show()

In [ ]:
# Analyze the time series data for stationarity and seasonality
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

print("Time Series Analysis for Model Selection:")
print("-" * 40)

# 1. Check for stationarity using ADF Test
# Null hypothesis: time series is non-stationary
adf_result = adfuller(y_train.dropna())
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print("Critical Values:")
for key, value in adf_result[4].items():
    print(f"\t{key}: {value:.4f}")
print(
    f"Series is {'stationary' if adf_result[1] < 0.05 else 'non-stationary'} according to ADF test"
)

# 2. Decompose the time series to see trend and seasonality
try:
    decomposition = seasonal_decompose(y_train.dropna(), model="additive", period=12)

    fig = plt.figure(figsize=(12, 10))
    fig.suptitle("Time Series Decomposition", fontsize=16)

    # Original
    ax1 = plt.subplot(411)
    ax1.plot(y_train, label="Original")
    ax1.legend(loc="upper left")

    # Trend
    ax2 = plt.subplot(412)
    ax2.plot(decomposition.trend, label="Trend")
    ax2.legend(loc="upper left")

    # Seasonality
    ax3 = plt.subplot(413)
    ax3.plot(decomposition.seasonal, label="Seasonality")
    ax3.legend(loc="upper left")

    # Residuals
    ax4 = plt.subplot(414)
    ax4.plot(decomposition.resid, label="Residuals")
    ax4.legend(loc="upper left")

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

    print("\nSeasonality Analysis:")
    print(
        f"Seasonal component range: {decomposition.seasonal.min():.2f} to {decomposition.seasonal.max():.2f}"
    )
    print(
        f"Seasonality strength (relative to residuals): {decomposition.seasonal.std() / decomposition.resid.std():.2f}"
    )

except Exception as e:
    print(f"Could not perform seasonal decomposition: {e}")

In [ ]:
from train.train_arima import train_and_forecast

forecast, actual, _, long = train_and_forecast(data_file)

# Re-define variables expected by visualization cells
arima_sf = forecast.rename("AutoARIMA")
mae_arima = mean_absolute_error(actual, forecast)


In [ ]:
cmp = pd.concat([future["price"], arima_sf], axis=1)
print("\n--- Actual vs Forecast (debug view) ---")
print(cmp.round(2))

print(f"MAE ARIMA: {mae_arima:.3f}")
print(f"ARIMA Accuracy = {100 - (mae_arima / y_test.mean() * 100):.2f}%")

### Plot The predicted data compare to an actual data

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(long.index, long["price"], label="Actual Prices", color="blue")
plt.plot(arima_sf.index, arima_sf, label="ARIMA Forecast", color="orange")
plt.axvline(
    x=pd.Timestamp(f"{gregorian_2567}-01-01"),
    color="red",
    linestyle="--",
    label="Forecast Start",
)
plt.xlabel("Date")
plt.ylabel("Price")
plt.title(f"{data_file.parent.name.capitalize()} Price Forecast with ARIMA")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig(fig_model_forecast / "ARIMA" / f"{data_file.parent.name}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Try multiple ARIMA variants to compare performance
from statsforecast.models import ARIMA

# Define models with better error handling
try:
    sf_multi = StatsForecast(
        models=[
            AutoARIMA(season_length=12, approximation=False),
            # Try simpler manual ARIMA models
            ARIMA(p=1, d=1, q=0, season_length=12),  # Simple model
            ARIMA(p=1, d=1, q=1, season_length=12),  # Standard (p,d,q)=(1,1,1)
            ARIMA(p=2, d=1, q=2, season_length=12),  # More complex
        ],
        freq="MS",
    )

    # Create a new dataframe to avoid interference with existing model
    sf_data_multi = (
        y_train.rename("y")
        .to_frame()
        .assign(ds=lambda df: df.index, unique_id="cassava")
        .reset_index(drop=True)
    )

    print("Fitting multiple ARIMA models for comparison...")
    sf_multi_forecasts = sf_multi.fit_predict(df=sf_data_multi, h=12)

    # Compare results of different models
    results = {}
    plt.figure(figsize=(14, 7))
    plt.plot(long.index, long["price"], label="Actual Prices", color="blue", alpha=0.7)

    # Get all column names except 'ds' and 'unique_id'
    model_columns = [
        col for col in sf_multi_forecasts.columns if col not in ["ds", "unique_id"]
    ]

    for model_name in model_columns:
        model_forecast = sf_multi_forecasts.set_index("ds")[model_name]
        mae = mean_absolute_error(y_test, model_forecast)
        accuracy = 100 - (mae / y_test.mean() * 100)
        results[model_name] = {"MAE": mae, "Accuracy": accuracy}

        # Plot each model forecast
        plt.plot(
            model_forecast.index,
            model_forecast.values,
            label=f"{model_name} (MAE: {mae:.3f})",
            linestyle="--",
        )

    plt.axvline(
        x=pd.Timestamp(f"{gregorian_2567}-01-01"),
        color="red",
        linestyle="--",
        label="Forecast Start",
    )
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.title("Cassava Price Forecast - Multiple ARIMA Models Comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Print comparison results
    print("\n--- Model Comparison Results ---")
    for model, metrics in results.items():
        print(f"{model}:")
        print(f"  MAE: {metrics['MAE']:.3f}")
        print(f"  Accuracy: {metrics['Accuracy']:.2f}%")
        print("-" * 30)

    # Identify the best model based on MAE
    best_model = min(results.items(), key=lambda x: x[1]["MAE"])
    print(
        f"\nBest model: {best_model[0]} with MAE: {best_model[1]['MAE']:.3f} and Accuracy: {best_model[1]['Accuracy']:.2f}%"
    )

except Exception as e:
    print(f"Error in multi-model comparison: {e}")
    print(
        "Try running the models individually to identify which one is causing the issue."
    )

In [ ]:
plt.figure(figsize=(14, 8))

# Calculate error values (in case this cell is run before the error calculation cell)
error = cmp["price"] - cmp["AutoARIMA"]
abs_error = abs(error)

if "error" not in cmp.columns:
    cmp["error"] = error
if "abs_error" not in cmp.columns:
    cmp["abs_error"] = abs_error

month_labels = [date.strftime("%b %Y") for date in cmp.index]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

ax1.fill_between(range(len(cmp)), cmp["abs_error"], alpha=0.7, color="red", label="Absolute Error")
ax1.plot(range(len(cmp)), cmp["abs_error"], color="darkred", linewidth=2, marker="o")
ax1.set_title(
    f"Monthly Absolute Errors - Area Under Curve ({data_file.parent.name.capitalize()})",
    fontsize=14, fontweight="bold",
)
ax1.set_xlabel("Month")
ax1.set_ylabel("Absolute Error")
ax1.set_xticks(range(len(cmp)))
ax1.set_xticklabels(month_labels, rotation=45)
ax1.grid(True, alpha=0.3)
ax1.legend()

total_area = np.trapezoid(cmp["abs_error"])
ax1.text(0.02, 0.98, f"Total Area Under Curve: {total_area:.2f}",
         transform=ax1.transAxes, fontsize=12, verticalalignment="top",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

ax2.plot(range(len(cmp)), cmp["price"], label="Actual Prices", color="blue", linewidth=2, marker="s")
ax2.plot(range(len(cmp)), cmp["AutoARIMA"], label="Forecast Prices", color="orange", linewidth=2, marker="o")

intersect_indices = []
for i in range(1, len(cmp)):
    prev_diff = cmp["price"].iloc[i - 1] - cmp["AutoARIMA"].iloc[i - 1]
    curr_diff = cmp["price"].iloc[i] - cmp["AutoARIMA"].iloc[i]
    if (prev_diff * curr_diff <= 0) and (prev_diff != 0 or curr_diff != 0):
        t = 0 if prev_diff == curr_diff else abs(prev_diff) / (abs(prev_diff) + abs(curr_diff))
        intersect_x = (i - 1) + t
        intersect_y = cmp["price"].iloc[i - 1] + t * (cmp["price"].iloc[i] - cmp["price"].iloc[i - 1])
        intersect_indices.append((intersect_x, intersect_y))

ax2.fill_between(range(len(cmp)), cmp["price"], cmp["AutoARIMA"],
                 where=(cmp["price"] >= cmp["AutoARIMA"]),
                 color="green", alpha=0.3, label="Under-prediction Area", interpolate=True)
ax2.fill_between(range(len(cmp)), cmp["price"], cmp["AutoARIMA"],
                 where=(cmp["price"] < cmp["AutoARIMA"]),
                 color="red", alpha=0.3, label="Over-prediction Area", interpolate=True)

for idx, (x, y) in enumerate(intersect_indices):
    ax2.axvline(x, color="purple", linestyle="--", alpha=0.7, linewidth=1.5)
    ax2.plot(x, y, "o", color="purple", markersize=8, label="Intersection Point" if idx == 0 else "")
    ax2.text(x, y * 1.05, f"Transition\n({month_labels[int(x)][:3]})",
             ha="center", color="purple", fontweight="bold",
             bbox=dict(facecolor="white", alpha=0.7, boxstyle="round,pad=0.3"))

ax2.set_title(
    f"Actual vs Forecast Prices with Error Areas ({data_file.parent.name.capitalize()})",
    fontsize=14, fontweight="bold",
)
ax2.set_xlabel("Month")
ax2.set_ylabel("Price")
ax2.set_xticks(range(len(cmp)))
ax2.set_xticklabels(month_labels, rotation=45)
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper right")

under_pred_area = np.trapezoid(np.maximum(cmp["price"] - cmp["AutoARIMA"], 0))
over_pred_area = np.trapezoid(np.maximum(cmp["AutoARIMA"] - cmp["price"], 0))
num_transitions = len(intersect_indices)
stats_text = (
    f"Under-prediction Area: {under_pred_area:.2f}\n"
    f"Over-prediction Area: {over_pred_area:.2f}\n"
    f"Number of Transitions: {num_transitions}"
)
ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes, fontsize=12, verticalalignment="top",
         bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.8))

plt.tight_layout()
plt.savefig(fig_model_error / "ARIMA" / f"{data_file.parent.name}_error.png", dpi=150, bbox_inches="tight")
plt.show()